In [26]:
# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [27]:
# ==========================================================
# Load Engineered Datasets
# ==========================================================

master_df = pd.read_csv(
    "../data/final/flight_operations_cross_dataset_feature_engineered.csv"
)

weather_df = pd.read_csv(
    "../data/final/weather_cross_dataset_feature_engineered.csv"
)

delay_df = pd.read_csv(
    "../data/final/delay_cross_dataset_feature_engineered.csv"
)

print("Master Dataset :", master_df.shape)
print("Weather Dataset:", weather_df.shape)
print("Delay Dataset  :", delay_df.shape)

Master Dataset : (7516, 77)
Weather Dataset: (1629108, 21)
Delay Dataset  : (3000000, 44)


In [28]:
# ==========================================================
# Global Operational Context
# ==========================================================

weather_context = {

    "High_Weather_Risk":

    round(
        (weather_df["Weather_Operational_Severity"]=="High").mean()*100,
        2
    ),

    "Immediate_Weather_Alerts":

    round(
        (weather_df["Weather_Alert_Priority"]=="Immediate").mean()*100,
        2
    )

}

delay_context = {

    "High_Delay_Risk":

    round(
        (delay_df["Delay_Operational_Severity"]=="High").mean()*100,
        2
    ),

    "Immediate_Delay_Alerts":

    round(
        (delay_df["Delay_Alert_Priority"]=="Immediate").mean()*100,
        2
    )

}

operational_context = {

    **weather_context,

    **delay_context

}

print(operational_context)

{'High_Weather_Risk': np.float64(0.14), 'Immediate_Weather_Alerts': np.float64(0.9), 'High_Delay_Risk': np.float64(8.71), 'Immediate_Delay_Alerts': np.float64(8.71)}


In [29]:
# ==========================================================
# Flight Operational Priority Scoring
# ==========================================================

complexity_score = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

airport_score = {
    "Local Airport": 1,
    "General Airport": 2,
    "Regional Hub": 3,
    "Major Hub": 4
}

aircraft_score = {
    "Light Capability": 1,
    "Standard Capability": 2,
    "Medium Capability": 3,
    "High Capability": 4,
    "Unknown": 2
}

distance_score = {
    "Remote": 1,
    "Moderate": 2,
    "Close": 3,
    "Very Close": 4
}

In [30]:
master_df["Flight_Operational_Score"] = (

    master_df["Operational_Complexity_Level"].map(complexity_score)

    +

    master_df["Airport_Operational_Profile"].map(airport_score)

    +

    master_df["Aircraft_Operational_Capability"].map(aircraft_score)

    +

    master_df["Distance_Risk_Level"].map(distance_score)

)

master_df["Flight_Operational_Score"].describe()

count    7516.000000
mean        8.605375
std         2.258643
min         4.000000
25%         7.000000
50%         8.000000
75%        10.000000
max        15.000000
Name: Flight_Operational_Score, dtype: float64

In [31]:
def operational_priority(score):

    if score >= 13:
        return "Critical"

    elif score >= 10:
        return "High"

    elif score >= 7:
        return "Medium"

    else:
        return "Low"


master_df["Flight_Operational_Priority"] = (

    master_df["Flight_Operational_Score"]

    .apply(operational_priority)

)

master_df["Flight_Operational_Priority"].value_counts()

Flight_Operational_Priority
Medium      3836
High        1907
Low         1365
Critical     408
Name: count, dtype: int64

In [32]:
# ==========================================================
# Weather Operational Status
# ==========================================================

if operational_context["High_Weather_Risk"] >= 5 or \
   operational_context["Immediate_Weather_Alerts"] >= 5:

    weather_status = "Severe"

elif operational_context["High_Weather_Risk"] >= 2 or \
     operational_context["Immediate_Weather_Alerts"] >= 2:

    weather_status = "Moderate"

else:
    weather_status = "Normal"

print("Weather Status :", weather_status)

Weather Status : Normal


In [33]:
# ==========================================================
# Delay Operational Status
# ==========================================================

if operational_context["High_Delay_Risk"] >= 15 or \
   operational_context["Immediate_Delay_Alerts"] >= 15:

    delay_status = "Severe"

elif operational_context["High_Delay_Risk"] >= 8 or \
     operational_context["Immediate_Delay_Alerts"] >= 8:

    delay_status = "Moderate"

else:
    delay_status = "Normal"

print("Delay Status :", delay_status)

Delay Status : Moderate


In [34]:
# ==========================================================
# System Operational Status
# ==========================================================

if weather_status == "Severe" or delay_status == "Severe":

    system_status = "Critical"

elif weather_status == "Moderate" or delay_status == "Moderate":

    system_status = "Busy"

else:

    system_status = "Normal"

print("System Status :", system_status)

System Status : Busy


In [35]:
# ==========================================================
# Smart Alert Prioritization
# ==========================================================

def smart_alert_priority(priority, complexity, system_status):

    # Critical flights
    if priority == "Critical":
        return "Immediate"

    # High priority flights
    elif priority == "High":

        if complexity == "High":
            return "Immediate"

        elif system_status == "Critical":
            return "Immediate"

        else:
            return "High"

    # Medium priority flights
    elif priority == "Medium":

        if complexity == "High" and system_status in ["Busy", "Critical"]:
            return "High"

        elif complexity == "Medium" and system_status == "Critical":
            return "High"

        else:
            return "Medium"

    # Low priority flights
    else:

        if complexity == "High" and system_status == "Critical":
            return "Medium"

        else:
            return "Low"

In [36]:
master_df["Smart_Alert_Priority"] = master_df.apply(
    lambda row: smart_alert_priority(
        row["Flight_Operational_Priority"],
        row["Operational_Complexity_Level"],
        system_status
    ),
    axis=1
)

master_df["Smart_Alert_Priority"].value_counts()

Smart_Alert_Priority
Medium       3811
High         1380
Low          1365
Immediate     960
Name: count, dtype: int64

In [37]:
# ==========================================================
# Information Filtering
# ==========================================================

def information_filter(priority):

    if priority == "Immediate":
        return "Display Immediately"

    elif priority == "High":
        return "High Visibility"

    elif priority == "Medium":
        return "Standard Monitoring"

    else:
        return "Background Monitoring"


master_df["Information_Filter_Level"] = (
    master_df["Smart_Alert_Priority"]
    .apply(information_filter)
)

master_df["Information_Filter_Level"].value_counts()

Information_Filter_Level
Standard Monitoring      3811
High Visibility          1380
Background Monitoring    1365
Display Immediately       960
Name: count, dtype: int64

In [38]:
def decision_support(alert):

    if alert == "Immediate":
        return "Immediate OCC Intervention"

    elif alert == "High":
        return "Enhanced Operational Monitoring"

    elif alert == "Medium":
        return "Routine Monitoring"

    else:
        return "Background Monitoring"

In [39]:
master_df["Decision_Support_Recommendation"] = (
    master_df["Smart_Alert_Priority"]
    .apply(decision_support)
)

master_df["Decision_Support_Recommendation"].value_counts()

Decision_Support_Recommendation
Routine Monitoring                 3811
Enhanced Operational Monitoring    1380
Background Monitoring              1365
Immediate OCC Intervention          960
Name: count, dtype: int64

In [40]:
# ==========================================================
# Save Final Information Overload Mitigation Dataset
# ==========================================================

master_df.to_csv(
    "../data/final/information_overload_mitigation_dataset.csv",
    index=False
)

print("Information Overload Mitigation Dataset Saved Successfully!")
print(master_df.shape)

Information Overload Mitigation Dataset Saved Successfully!
(7516, 82)
